# NHANES project about Periodontal disease and Geriatric Nutrition Risk Index (GNRI): descriptive and regression analysis
> This notebook has the purpose to collect all the analysis on Nhanes dataset for a medical paper project 

Requirements and Information:
1. Nhanes dataset from 2009/10 to 2013/14
2. Outcome:
    - Geriatric Nutrition Risk Index (GNRI)
3. Exposure:
    - number of teeth (OHXDEN)
    - consists of 2 categories: patients with < 20 teeth, patients with >= 20 teeth
    - Other categorization:
        1. Edentulus : 0 teeth
        2. Severe Loss : 1-9 teeth
        3. Moderate Loss : 10-19 teeth
        4. Nearly Complete : >=20 teeth
4. Confounding Variables:
    - Gender (RIAGENDR)
    - Age at screening (RIDAGEYR)
    - Race (RIDRETH1)
    - Education	(DMDEDUC2)
    - Poverty income ratio (INDFMPIR)
    - Smoking status (SMQ020)
    - Alchool intake (ALQ101)
5. Mediators:
    - Heart failure	(RIDRETH1)  
    - Coronary heart disease (MCQ160b)
    - Stroke (MCQ160c)
    - Liver disease	(MCQ160o)
    - Cancer (MCQ220)
    - Diabetes (DIQ010)
    - High blood pressure (BPQ020)
6. Age => 60

## Import Libraries

In [ ]:
library(haven)
library(nhanesA)
library(survey)
library(MASS)
library(dplyr)
library(tidyr)
library(tidyverse)
library(ggplot2)
library(readr)
library(flextable)
library(officer)
library(nnet)
library(broom)
library(ggplot2)
library(patchwork)

## Configurations

In [ ]:
path_to_data_09_10 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2009_10/"
path_to_data_11_12 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2011_12/"
path_to_data_13_14 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2013_14/"

## Load Dataset & Feature Selection

In [ ]:
# Datasets for 2009/10 period

demo_09_10 <- read_xpt(file.path(path_to_data_09_10, "DEMO_F.xpt"))

demo_09_10_selected <- demo_09_10 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_09_10 <- read_xpt(file.path(path_to_data_09_10, "ALQ_F.xpt"))

alcohol_09_10_selected <- alcohol_09_10 %>%
  select(SEQN, ALQ101)

smoking_09_10 <- read_xpt(file.path(path_to_data_09_10, "SMQ_F.xpt.txt"))

smoking_09_10_selected <- smoking_09_10 %>%
    select(SEQN, SMQ020)

med_conditions_09_10 <- read_xpt(file.path(path_to_data_09_10, "MCQ_F.xpt"))

med_conditions_09_10_selected <- med_conditions_09_10 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_09_10 <- read_xpt(file.path(path_to_data_09_10, "BPQ_F.xpt"))

blood_pressure_09_10_selected <- blood_pressure_09_10 %>%
    select(SEQN, BPQ020)


diabetes_09_10 <- read_xpt(file.path(path_to_data_09_10, "DIQ_F.xpt"))

diabetes_09_10_selected <- diabetes_09_10 %>%
    select(SEQN, DIQ010)


teeth_09_10 <- read_xpt(file.path(path_to_data_09_10, "OHXDEN_F.xpt.txt"))

selected_cols <- colnames(teeth_09_10)[grepl("^OHX\\d{2}TC", colnames(teeth_09_10))]

teeth_09_10_selected <- teeth_09_10 %>%
    select(SEQN, all_of(selected_cols))


albumin_09_10 <- read_xpt(file.path(path_to_data_09_10, "BIOPRO_F.xpt.txt"))

albumin_09_10_selected <- albumin_09_10 %>%
    select(SEQN, LBDSALSI)

w_h_09_10 <- read_xpt(file.path(path_to_data_09_10, "BMX_F.xpt"))

w_h_09_10_selected <- w_h_09_10 %>%
    select(SEQN, BMXWT, BMXHT)

In [ ]:
# Datasets for 2011/12 period

demo_11_12 <- read_xpt(file.path(path_to_data_11_12, "DEMO_G.xpt.txt"))

demo_11_12_selected <- demo_11_12 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_11_12 <- read_xpt(file.path(path_to_data_11_12, "ALQ_G.xpt.txt"))

alcohol_11_12_selected <- alcohol_11_12 %>%
  select(SEQN, ALQ101)


smoking_11_12 <- read_xpt(file.path(path_to_data_11_12, "SMQ_G.xpt.txt"))

smoking_11_12_selected <- smoking_11_12 %>%
    select(SEQN, SMQ020)


med_conditions_11_12 <- read_xpt(file.path(path_to_data_11_12, "MCQ_G.xpt.txt"))

med_conditions_11_12_selected <- med_conditions_11_12 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_11_12 <- read_xpt(file.path(path_to_data_11_12, "BPQ_G.xpt.txt"))

blood_pressure_11_12_selected <- blood_pressure_11_12 %>%
    select(SEQN, BPQ020)


diabetes_11_12 <- read_xpt(file.path(path_to_data_11_12, "DIQ_G.xpt.txt"))

diabetes_11_12_selected <- diabetes_11_12 %>%
    select(SEQN, DIQ010)


teeth_11_12 <- read_xpt(file.path(path_to_data_11_12, "OHXDEN_G.xpt.txt"))

selected_cols <- colnames(teeth_11_12)[grepl("^OHX\\d{2}TC", colnames(teeth_11_12))]

teeth_11_12_selected <- teeth_11_12 %>%
    select(SEQN, all_of(selected_cols))


albumin_11_12 <- read_xpt(file.path(path_to_data_11_12, "BIOPRO_G.xpt.txt"))

albumin_11_12_selected <- albumin_11_12 %>%
    select(SEQN, LBDSALSI)

w_h_11_12 <- read_xpt(file.path(path_to_data_11_12, "BMX_G.xpt.txt"))

w_h_11_12_selected <- w_h_11_12 %>%
    select(SEQN, BMXWT, BMXHT)

In [ ]:
# Datasets for 2013/14 period

demo_13_14 <- read_xpt(file.path(path_to_data_13_14, "DEMO_H.xpt.txt"))

demo_13_14_selected <- demo_13_14 %>%
    select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)


alcohol_13_14 <- read_xpt(file.path(path_to_data_13_14, "ALQ_H.xpt.txt"))

alcohol_13_14_selected <- alcohol_13_14 %>%
    select(SEQN, ALQ101)


smoking_13_14 <- read_xpt(file.path(path_to_data_13_14, "SMQ_H.xpt.txt"))

smoking_13_14_selected <- smoking_13_14 %>%
    select(SEQN, SMQ020)


med_conditions_13_14 <- read_xpt(file.path(path_to_data_13_14, "MCQ_H.xpt.txt"))

med_conditions_13_14_selected <- med_conditions_13_14 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_13_14 <- read_xpt(file.path(path_to_data_13_14, "BPQ_H.xpt.txt"))

blood_pressure_13_14_selected <- blood_pressure_13_14 %>%
    select(SEQN, BPQ020)


diabetes_13_14 <- read_xpt(file.path(path_to_data_13_14, "DIQ_H.xpt.txt"))

diabetes_13_14_selected <- diabetes_13_14 %>%
    select(SEQN, DIQ010)


teeth_13_14 <- read_xpt(file.path(path_to_data_13_14, "OHXDEN_H.xpt.txt"))

selected_cols <- colnames(teeth_13_14)[grepl("^OHX\\d{2}TC", colnames(teeth_13_14))]

teeth_13_14_selected <- teeth_13_14 %>%
    select(SEQN, all_of(selected_cols))


albumin_13_14 <- read_xpt(file.path(path_to_data_13_14, "BIOPRO_H.xpt.txt"))

albumin_13_14_selected <- albumin_13_14 %>%
    select(SEQN, LBDSALSI)

w_h_13_14 <- read_xpt(file.path(path_to_data_13_14, "BMX_H.xpt.txt"))

w_h_13_14_selected <- w_h_13_14 %>%
    select(SEQN, BMXWT, BMXHT)

## Merge datasets without NA and missing values
> Merge all data from each datasets and then exclude patients

In [ ]:
# Merge datasets demographics and intrinsic capacity data

datasets_09_10 <- list(
  demo_09_10_selected, alcohol_09_10_selected, smoking_09_10_selected, med_conditions_09_10_selected,
  blood_pressure_09_10_selected, diabetes_09_10_selected,
  albumin_09_10_selected, w_h_09_10_selected, teeth_09_10_selected
)

datasets_11_12 <- list(
  demo_11_12_selected, alcohol_11_12_selected, smoking_11_12_selected, med_conditions_11_12_selected,
  blood_pressure_11_12_selected, diabetes_11_12_selected,
  albumin_11_12_selected, w_h_11_12_selected, teeth_11_12_selected
)

datasets_13_14 <- list(
  demo_13_14_selected, alcohol_13_14_selected, smoking_13_14_selected, med_conditions_13_14_selected,
  blood_pressure_13_14_selected, diabetes_13_14_selected,
  albumin_13_14_selected, w_h_13_14_selected, teeth_13_14_selected
)

# Horizontal union for period 2009/10, 2011/12, 2013/14

df_09_10 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_09_10)

df_11_12 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_11_12)

df_13_14 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_13_14)

# Vertical union

df_final <- bind_rows(df_09_10, df_11_12, df_13_14)

print("Dimensions before removing NA values")
dim(df_final)

# Filter with AGE >= 60

df_final_age_60 <- subset(df_final, RIDAGEYR >= 60)

print("Dimensions with AGE >= 60")
dim(df_final_age_60)

# Excluding patients with missing values in features required for GNRI calculation (weight, height, albumin)

df_final_excluding_GNRI <- df_final_age_60[complete.cases(df_final_age_60[, c('BMXWT', 'BMXHT', 'LBDSALSI')]), ]

print("Dimensions without GNRI missing values")
dim(df_final_excluding_GNRI)

# Excluding patients with no examinations for Teeth counts

df_final_excluding_teeth <- df_final_excluding_GNRI %>%
  filter(rowSums(!is.na(select(., starts_with("OHX")))) > 0)

print("Dimensions without Teeth counts missing values")
dim(df_final_excluding_teeth)

teeth_cols <- grep("^OHX\\d{2}TC$", names(df_final_excluding_teeth), value = TRUE)
df_final_excluding_teeth <- df_final_excluding_teeth %>%
  filter(!if_any(all_of(teeth_cols), ~ . == 1))

# Excluding patients with missing values in Confounding features

df_final_excluding_confounding <- df_final_excluding_teeth[complete.cases(df_final_excluding_teeth[, 
                                  c('RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'ALQ101', 'SMQ020', 'MCQ160B',
                                  'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ160L', 'MCQ220', 'BPQ020', 'DIQ010')]), ]

df_final_merged <- df_final_excluding_confounding %>%
  filter(!if_any(c(DMDEDUC2, ALQ101, SMQ020, MCQ160B, MCQ160C, MCQ160D,
                   MCQ160E, MCQ160F, MCQ160L, MCQ220, BPQ020, DIQ010), ~ . == 9))

df_final_merged <- df_final_merged %>%
  filter(!if_any(c(DMDEDUC2, ALQ101, SMQ020, MCQ160B, MCQ160C, MCQ160D,
                   MCQ160E, MCQ160F, MCQ160L, MCQ220, BPQ020, DIQ010), ~ . == 7))

print("Dimensions without Confounding missing values")
dim(df_final_merged)

In [ ]:
# for cycle to see table for each categorical variable
categorical_vars <- c('RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'ALQ101', 'SMQ020', 
                      'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 
                      'MCQ160F', 'MCQ160L', 'MCQ220',
                      'BPQ020', 'DIQ010')
for (var in categorical_vars) {
  print(paste("Table for", var))
  print(table(df_final_merged[[var]], useNA = "ifany"))
}

In [ ]:
teeth_cols <- grep("^OHX\\d{2}TC$", names(df_final_merged), value = TRUE)

for (var in teeth_cols) {
  print(paste("Table for", var))
  print(table(df_final_merged[[var]], useNA = "ifany"))
}

In [ ]:
# check how many patients have one or more teeth_cols with value 1 - Technically should be 0

find_patients_with_1 <- function(df) {
  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)
  
  no_teeth_patients <- df[rowSums(df[, teeth_cols] == 1, na.rm = TRUE) > 0, ]
  
  return(no_teeth_patients)
}

patients_with_value_1 <- find_patients_with_1(df_final_merged)
dim(patients_with_value_1)

In [ ]:
# Saving completed and cleaned dataframe for analysis

write.csv(df_final_merged, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/merged_and_cleaned_df_09_14_GNRI_teeth.csv", row.names = FALSE)

## Teeth counts

Preprocessed features:
- total number of teeth
- binary category: >=20 teeth or < 20 teeth
- edentulus category
- Other categorization:
    1. Edentulus : 0 teeth
    2. Severe Loss : 1-9 teeth
    3. Moderate Loss : 10-19 teeth
    4. Nearly Complete : >=20 teeth

In [ ]:
df_final_merged <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/merged_and_cleaned_df_09_14_GNRI_teeth.csv")

head(df_final_merged)

In [ ]:
# Functions to check if there are patients with zero permanent teeth,
# and moreover, patients with only not present teeth and fragments/root

find_patients_no_teeth <- function(df) {

  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)
  
  no_teeth_patients <- df[rowSums(df[, teeth_cols] == 2, na.rm = TRUE) == 0, ]
  
  return(no_teeth_patients)
}

find_patients_with_non4_values <- function(df) {
  
  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)

  no_teeth_patients <- find_patients_no_teeth(df)

  print("Dimensions of patients with no teeth")
  print(dim(no_teeth_patients))

  patients_with_non4 <- no_teeth_patients[rowSums(no_teeth_patients[, teeth_cols] == 4, na.rm = TRUE) == 0, ]

  if (dim(patients_with_non4)[1] != 0) {
     print("Dimensions of patients with non-4 values in teeth columns")
     print(dim(patients_with_non4))
  } else {
     print("Dimensions of patients with non-4 values in teeth columns are empty")
  }

  return(patients_with_non4)

}

test_edentolus_no_4 <- find_patients_with_non4_values(df_final_merged)

test_edentolus <- find_patients_no_teeth(df_final_merged)

# funzione per sapere per ogni paziente quanti denti hanno valori differenti da 4 (no table)

i = 1
j <- 0
for (patient in test_edentolus$SEQN) {
   teeth_cols <- grep("^OHX\\d{2}TC$", names(test_edentolus), value = TRUE)
   
   non4_count <- rowSums(test_edentolus[i, teeth_cols] != 4, na.rm = TRUE)

   if (non4_count > 0) {
      print(paste("Patient SEQN:", patient, "has", non4_count, "teeth with non-4 values."))
      j <- j + 1
   }
   i <- i + 1
}
print(paste("Total patients with non-4 teeth values:", j))

In [ ]:
# Function to calculate total number of teeth for each patient and categorize it

count_teeth <- function(df) {

  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)
  
  df$total_teeth <- rowSums(df[, teeth_cols] == 2, na.rm = TRUE)
  
  # Binary category: 1 if >=20 teeth, 0 otherwise
  df$has_20_or_more_teeth <- ifelse(df$total_teeth >= 20, 1, 0)
  
  # edentulous patients (every 32 teeth with value 4 or 5 or 3)

  df$edentulous <- ifelse(rowSums(df[, teeth_cols] == 4 | df[, teeth_cols] == 5 | df[, teeth_cols] == 3, na.rm = TRUE) == length(teeth_cols), 1, 0)
  
  # Other possible categories: Edentulous, Severe, Moderate, Nearly Complete
  df$teeth_category <- cut(
    df$total_teeth,
    breaks = c(-Inf, 0, 9, 19, 32),
    labels = c("Edentulous", "Severe Loss (1-9)", "Moderate Loss (10-19)", "Nearly Complete (20-32)"),
    right = TRUE
  )
  
  df <- df[, !names(df) %in% teeth_cols]
  
  return(df)
}

df <- as.data.frame(count_teeth(df_final_merged))
head(df)
dim(df)
table(df$teeth_category, useNA = "ifany")

In [ ]:
# Saving preprocessed

write.csv(df, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_GNRI_teeth_09_14.csv", row.names=FALSE)

## GNRI formula

In [ ]:
df <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_GNRI_teeth_09_14.csv")

head(df)

In [ ]:
calculate_gnri <- function(df) {
  df$ideal_weight <- ifelse(
    df$RIAGENDR == 1,
    0.75 * df$BMXHT - 62.5,
    0.60 * df$BMXHT - 40
  )

  # Weight ratio (bounded at 1)
  weight_ratio <- pmin(df$BMXWT / df$ideal_weight, 1)

  # GNRI
  df$gnri_score <- (1.489 * df$LBDSALSI) + (41.7 * weight_ratio)

  # Multi-class category
  df$gnri_category <- cut(
    df$gnri_score,
    breaks = c(-Inf, 82, 91, 98, Inf),
    labels = c("Severe risk", "Moderate risk", "Low risk", "No risk"),
    right = TRUE
  )

  # Binary category
  df$gnri_binary <- ifelse(df$gnri_score < 98, "Low-GNRI", "High-GNRI")

  return(df)
}

In [ ]:
df_with_gnri <- calculate_gnri(df)

head(df_with_gnri[c("ideal_weight", "gnri_score", "gnri_category", "gnri_binary")])

In [ ]:
table(df_with_gnri$gnri_binary)

In [ ]:
# Histogram and violin plot for GNRI score

options(repr.plot.width = 12, repr.plot.height = 6)

df_with_gnri$gender <- factor(df_with_gnri$RIAGENDR, levels = c(1, 2), labels = c("Male", "Female"))

hist_plot <- ggplot(df_with_gnri, aes(x = gnri_score)) +
  geom_histogram(binwidth = 5, fill = "#0073C2FF", color = "white", alpha = 0.8) +
  labs(title = "GNRI Score Distribution", x = "GNRI Score", y = "Count") +
  theme_minimal()

violin_plot <- ggplot(df_with_gnri, aes(x = gender, y = gnri_score, fill = gender)) +
  geom_violin(trim = FALSE, alpha = 0.6) +
  geom_boxplot(width = 0.1, outlier.shape = NA) +
  labs(title = "GNRI Score by Gender", x = "Gender", y = "GNRI Score") +
  scale_fill_manual(values = c("Male" = "#0073C2FF", "Female" = "#EFC000FF")) +
  theme_minimal()

hist_plot + violin_plot

In [ ]:
write.csv(df_with_gnri, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/final_preprocessed_df_GNRI_teeth_09_14.csv", row.names = FALSE)

## Descriptive Analysis

In [ ]:
# Load preprocessed and cleaned df

df_test <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/final_preprocessed_df_GNRI_teeth_09_14.csv",
               header = TRUE)

head(df_test)

### Select weights

In [ ]:
# Weights from Demographic datasets

demo_09_10 <- read_xpt(file.path(path_to_data_09_10, "DEMO_F.xpt"))

demo_09_10_weights <- demo_09_10 %>%
    select(SEQN, RIDAGEYR, WTMEC2YR, SDMVPSU, SDMVSTRA)

demo_11_12 <- read_xpt(file.path(path_to_data_11_12, "DEMO_G.xpt.txt"))

demo_11_12_weights <- demo_11_12 %>%
    select(SEQN, RIDAGEYR, WTMEC2YR, SDMVPSU, SDMVSTRA)

demo_13_14 <- read_xpt(file.path(path_to_data_13_14, "DEMO_H.xpt.txt"))

demo_13_14_weights <- demo_13_14 %>%
    select(SEQN, RIDAGEYR, WTMEC2YR, SDMVPSU, SDMVSTRA)

weights_09_10 <- list(
  demo_09_10_weights
)

weights_11_12 <- list(
  demo_11_12_weights
)

weights_13_14 <- list(
  demo_13_14_weights
)

# Horizontal union for period 2009/10, 2011/12, 2013/14

wt_09_10 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), weights_09_10)

wt_11_12 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), weights_11_12)

wt_13_14 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), weights_13_14)

# Vertical union

wt_final <- bind_rows(wt_09_10, wt_11_12, wt_13_14)

print("Dimensions before removing NA values")
dim(wt_final)

# Filter with AGE >= 60

wt_final_age_60 <- subset(wt_final, RIDAGEYR >= 60)

print("Dimensions with AGE >= 60")
dim(wt_final_age_60)

wt_final_age_60 <- subset(wt_final_age_60, select = -RIDAGEYR)
head(wt_final_age_60)

# Merge wt_final_age_60 with my final data frame 

df_final_merged <- df_test %>%
  inner_join(wt_final_age_60, by = "SEQN")

dim(df_final_merged)

In [ ]:
# Preprocessing for WTMEC2YR: divide it for the number of NHANES cycles used (3 for our case)

df_final_merged[, "wt"] = df_final_merged[, "WTMEC2YR"] / 3

head(df_final_merged)

In [ ]:
# Save df with weights

write.csv(df_final_merged, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_teeth_GNRI_09_14_with_weights.csv", row.names=FALSE)

In [ ]:
# Using srvyr to check population estimate
library(srvyr)

svy_obj <- df_final_merged %>%
  as_survey_design(
    ids = SDMVPSU,
    strata = SDMVSTRA,
    weights = wt,
    nest = TRUE
  )

pop_est <- svy_obj %>%
  summarize(pop = survey_total(1, vartype = "ci"))

print(pop_est)

### Multi-Label Descriptive

In [ ]:
# Load preprocessed and cleaned df

df_final_merged <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_teeth_GNRI_09_14_with_weights.csv",
               header = TRUE)

head(df_final_merged)

In [ ]:
df_final_merged$teeth_category = factor(df_final_merged$teeth_category,
                              levels = c("Nearly Complete (20-32)", "Moderate Loss (10-19)",
                                         "Severe Loss (1-9)", "Edentulous"),
                              labels = c("20 teeth or more", "10-19 teeth",
                                          "1-9 teeth", "Edentulous"))

table(df_final_merged$teeth_category)

In [ ]:
# Violin plot for GNRI score by periodontitis classification

library(ggpubr)

options(repr.plot.width = 15, repr.plot.height = 10)

comparisons <- list(
  c("20 teeth or more", "10-19 teeth"),
  c("20 teeth or more", "1-9 teeth"),
  c("20 teeth or more", "Edentulous")
  #c("10-19 teeth", "1-9 teeth"),
  #c("10-19 teeth", "Edentulous"),
  #c("1-9 teeth", "Edentulous")
)

ggplot(df_final_merged, aes(x = teeth_category, y = gnri_score, fill = teeth_category)) +
  geom_violin(trim = FALSE, alpha = 0.6) +
  geom_boxplot(width = 0.1, outlier.shape = NA, color = "black") +
  stat_compare_means(comparisons = comparisons, p.adjust.method = "bonferroni", method = "wilcox.test", label = "p.format", size = 5) +
  labs(title = "GNRI Score by Teeth Category",
       x = "Teeth Category", y = "GNRI Score") +
  scale_fill_manual(values = c("20 teeth or more" = "#0073C2FF", "10-19 teeth" = "#EFC000FF", "1-9 teeth" = "#CD534CFF", "Edentulous" = "#99F792")) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),
    axis.title = element_text(size = 16),
    axis.text = element_text(size = 14),
    legend.title = element_blank(),
    legend.text = element_text(size = 13),
    legend.position = "right"
  )

In [ ]:
# Recoding all features with consistent approach

df <- df_final_merged %>%
  mutate(
      
    # Demographic variables
    RIAGENDR = factor(RIAGENDR, levels = c(1, 2),
                    labels = c("Male", "Female")),
    
    DMDEDUC2 = factor(DMDEDUC2, levels = 1:5,
                     labels = c("Less than 9th grade", "9-11th grade",
                                "High school graduate",
                                "Some college/AA degree",
                                "College graduate or above"),
                     ordered = TRUE),
    
    RIDRETH1 = factor(RIDRETH1, levels = 1:5,
                      labels = c("Mexican American", "Other Hispanic",
                                "Non-Hispanic White", "Non-Hispanic Black",
                                "Other Race")),
    
    # Lifestyle variables - Set "No" as reference
    SMQ020 = factor(SMQ020, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Smoking status
    
    ALQ101 = factor(ALQ101, levels = c(2, 1),
                   labels = c("Under 12 drinks/1 yr", "Over 12 drinks/1 yr")),  # Alcohol
    
    # Medical conditions - all with "No" as reference level
    MCQ160B = factor(MCQ160B, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Heart failure
    
    MCQ160C = factor(MCQ160C, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Coronary heart disease
    
    MCQ160D = factor(MCQ160D, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Angina
    
    MCQ160E = factor(MCQ160E, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Heart attack
    
    MCQ160F = factor(MCQ160F, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Stroke
    
    MCQ220 = factor(MCQ220, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Cancer
    
    MCQ160L = factor(MCQ160L, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Liver condition
    
    BPQ020 = factor(BPQ020, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Hypertension
    
    # Multi-categorical - set "No" as reference
    DIQ010 = factor(DIQ010, levels = c(2, 1, 3),
                   labels = c("No", "Yes", "Borderline"))  # Diabetes
  )

# Verify transformations and check for any issues
summary(df[, c("RIAGENDR", "DMDEDUC2", "RIDRETH1", "SMQ020", "ALQ101", 
               "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220", "MCQ160L",
               "BPQ020", "DIQ010")])

In [ ]:
# for cycle to see categorical variables table with teeth category grouping n and %

for (var in c("RIAGENDR", "DMDEDUC2", "RIDRETH1", "SMQ020", "ALQ101", 
              "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", 
              "MCQ220", "MCQ160L", "BPQ020", "DIQ010")) {
  cat("\nVariable:", var, "\n")
  print(table(df[[var]], df$teeth_category), useNA = "ifany")
  print(prop.table(table(df[[var]], df$teeth_category), margin = 2))
}

In [ ]:
# Descriptive analysis with multi classes:
# "20 teeth or more", "10-19 teeth", "1-9 teeth", "Edentulous"

create_descriptive_table <- function(df, survey_design = NULL) {
  require(gtsummary)
  require(dplyr)
  require(survey)
  require(srvyr)
  
  # Check if survey design is provided
  use_survey_design <- !is.null(survey_design)
  
  # Normality test a priori for continuous variables
  continuous_vars <- c("RIDAGEYR", "INDFMPIR")
  
  normality_results <- list()
  
  for (var in continuous_vars) {
    # Limit: 5000 observations for Shapiro-Wilk test
    if (length(na.omit(df[[var]])) > 5000) {
      sample_data <- sample(na.omit(df[[var]]), 5000)
    } else {
      sample_data <- na.omit(df[[var]])
    }
    
    test_result <- shapiro.test(sample_data)
    normality_results[[var]] <- test_result$p.value > 0.05
    message(var, " p-value: ", test_result$p.value)
  }
  
  message("Normality test results:")
  for (var in names(normality_results)) {
    message(var, ": ", ifelse(normality_results[[var]], "Normal", "Non-normal"))
  }

  variables_to_include <- c(
                "gnri_score", "total_teeth", "RIDAGEYR", "INDFMPIR", "RIAGENDR",
                "DMDEDUC2", "RIDRETH1", "SMQ020", "ALQ101",
                "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220", "MCQ160L",
                "BPQ020", "DIQ010")

  # Variables using median (IQR) or mean (SD)
  median_vars <- names(normality_results)[!unlist(normality_results)]
  median_vars <- c(median_vars, "total_teeth")

  stat_labels <- list(
    "gnri_score" = "GNRI Score (mean, SD)",
    "total_teeth" = "Number of teeth (median, IQR)",
    "RIDAGEYR" = "Age (mean, SD)",
    "INDFMPIR" = "Ratio of family income (mean, SD)",
    "RIAGENDR" = "Gender (n, %)",
    "RIDRETH1" = "Ethnicity (n, %)",
    "DMDEDUC2" = "Education (n, %)",
    "SMQ020" = "Smoking (n, %)",
    "ALQ101" = "Alcohol intake (n, %)",
    "MCQ160B" = "Heart Failure (n, %)",
    "MCQ160C" = "Coronary Heart (n, %)",
    "MCQ160D" = "Angina (n, %)",
    "MCQ160E" = "Heart Attack (n, %)",
    "MCQ160F" = "Stroke (n, %)",
    "MCQ220" = "Cancer (n, %)",
    "MCQ160L" = "Liver (n, %)",
    "BPQ020" = "Hypertension (n, %)",
    "DIQ010" = "Diabetes (n, %)"
  )
  
  for (var in names(normality_results)) {
    if (normality_results[[var]]) {
      stat_labels[[var]] <- gsub("\\(median, IQR\\)", "(mean, SD)", stat_labels[[var]])
    } else {
      stat_labels[[var]] <- gsub("\\(mean, SD\\)", "(median, IQR)", stat_labels[[var]])
    }
  }

  # Define statistics
  stat_list <- list(
    all_continuous() ~ "{mean} ({sd})",
    all_categorical() ~ "{n} ({p}%)"
  )
  for (var in median_vars) {
    stat_list[[var]] <- "{median} ({p25}, {p75})"
  }
  
    # Create table with gtsummary - use tbl_summary for consistent display
    # but calculate weighted population for the additional column
    table_strat <- df %>%
      tbl_summary(
        by = teeth_category,
        include = all_of(variables_to_include),
        statistic = stat_list,
        label = stat_labels,
        missing = "ifany",
        missing_text = "Missing",
        digits = all_continuous() ~ 2,
        value = all_categorical() ~ "level",
        type = all_categorical() ~ "categorical"
      ) %>%
      add_p(
        test = list(
          continuous_vars[unlist(normality_results[continuous_vars])] ~ "anova", 
          continuous_vars[!unlist(normality_results[continuous_vars])] ~ "kruskal.test",
          `total_teeth` ~ "kruskal.test",
          all_categorical() ~ "chisq.test",
          DMDEDUC2 ~ "kruskal.test"  # it is an ordinal feature
        ),
        # some categories have few patients
        test.args = list(Ethnicity = list(simulate.p.value = TRUE))
      ) %>%
      add_overall()
  
  # Calculate N for each group
  total_n <- nrow(df)
  n_by_group <- df %>%
    group_by(teeth_category) %>%
    summarize(n = n()) %>%
    pull(n, name = teeth_category)
  
  # Add weighted N in millions column only if survey design is provided
  if (use_survey_design) {
    survey_obj <- survey_design %>% 
      as_survey(options = list(lonely.psu = "adjust"))

    total_pop_in_millions <- survey_obj %>%
      summarize(pop = survey_total(1)) %>%
      mutate(pop_millions = pop/1000000) %>%
      pull(pop_millions)

    pop_by_group_in_millions <- survey_obj %>%
      group_by(teeth_category) %>%
      summarize(pop = survey_total(1)) %>%
      mutate(pop_millions = pop/1000000)
    
    table_strat <- table_strat %>%
      modify_table_body(
        ~.x %>%
          dplyr::mutate(
            weighted_n = case_when(
              is.na(row_type) ~ "", 
              row_type == "label_header" ~ "**Weighted N (Millions)**",
              row_type == "label" ~ "",
              row_type == "level" & !is.na(variable) & !is.na(label) ~ "",
              TRUE ~ ""
            )
          ) %>%
          dplyr::relocate(weighted_n, .before = stat_0)
      )
    
    # Add weighted N column as an additional column
    table_strat <- table_strat %>%
      modify_header(
        label = "**Characteristics**",
        weighted_n = paste0("**Weighted N**\n**in Millions**")
      )
    
    # Add weighted population estimates for each row
    if (use_survey_design) {
      # Function to calculate weighted population for a specific variable and level
      calculate_weighted_pop <- function(var_name, level = NULL) {
        tryCatch({
          if (is.null(level)) {
            # For continuous variables: sum weights where variable is not NA
            var_data <- survey_design$variables[[var_name]]
            weights_sum <- sum(weights(survey_design, "analysis")[!is.na(var_data)]) / 1000000
            return(weights_sum)
          } else {
            # For categorical variables and specific levels
            var_data <- survey_design$variables[[var_name]]
            level_match <- var_data == level & !is.na(var_data)
            weights_sum <- sum(weights(survey_design, "analysis")[level_match]) / 1000000
            return(weights_sum)
          }
        }, error = function(e) {
          message("Error calculating weighted population for ", var_name, 
                  if(!is.null(level)) paste(" level:", level), ": ", e$message)
          return(NA)
        })
      }
      
      # Update each row with weighted population estimate
      table_strat$table_body <- table_strat$table_body %>%
        rowwise() %>%
        mutate(
          weighted_n = case_when(
            !is.na(row_type) & row_type == "label" & !is.na(variable) ~ 
              sprintf("%.2f", calculate_weighted_pop(variable)),
            !is.na(row_type) & row_type == "level" & !is.na(variable) & !is.na(label) ~
              sprintf("%.2f", calculate_weighted_pop(variable, label)),
            TRUE ~ weighted_n
          )
        ) %>%
        ungroup()
    }
  } else {
    # If no survey design, just modify the headers without weighted N
    table_strat <- table_strat %>%
      modify_header(
        label = "**Characteristics**",
        stat_0 = paste0("**Total**\nN = ", total_n), 
        stat_1 = paste0("**20 teeth or more**\nN = ", ifelse("20 teeth or more" %in% names(n_by_group), n_by_group["20 teeth or more"], 0)), 
        stat_2 = paste0("**10-19 teeth**\nN = ", ifelse("10-19 teeth" %in% names(n_by_group), n_by_group["10-19 teeth"], 0)),
        stat_3 = paste0("**1-9 teeth**\nN = ", ifelse("1-9 teeth" %in% names(n_by_group), n_by_group["1-9 teeth"], 0)), 
        stat_4 = paste0("**Edentulous**\nN = ", ifelse("Edentulous" %in% names(n_by_group), n_by_group["Edentulous"], 0)),
        p.value = "**P-value**"
      )
  }
  
  # Notes
  table_strat <- table_strat %>%
    modify_footnote(
      all_stat_cols() ~ "Values are n (%) for categorical variables, median (IQR) for non-normally distributed continuous variables, and mean (SD) for normally distributed continuous variables."
      #update = all_stat_cols() ~ "Values are n (%) for categorical variables, median (IQR) for non-normally distributed continuous variables, and mean (SD) for normally distributed continuous variables."
    )
  
  if (use_survey_design) {
    table_strat <- table_strat %>%
      modify_footnote(
        add = "Weighted N in millions represents the estimated US population based on NHANES survey weights."
      )
  }
  
  return(table_strat)
}

In [ ]:
# Function to create survey design for NHANES datasets (to update for case with 3 different two-years period)

create_nhanes_design <- function(df) {

  design <- svydesign(
    id = ~SDMVPSU,
    strata = ~SDMVSTRA,
    weights = ~wt,
    options = list(lonely.psu = "adjust"),
    nest = TRUE,
    data = df
  )
  return(design)
}

In [ ]:
# Saving weighted results in docx format

nhanes_design <- create_nhanes_design(df)

result_table_weighted <- create_descriptive_table(df, nhanes_design)
flex_table_weighted <- result_table_weighted %>% as_flex_table()

save_as_docx(flex_table_weighted,
            path = "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/results/NHANES_09_14_GNRI_teeth/multi_labels_weighted_descriptive_analysis.docx")

## Regression Analysis

### Load Preprocessed Data + Recoded outcome and some variables

In [ ]:
# Load preprocessed and cleaned df

df_final_merged <- read.csv("/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_teeth_GNRI_09_14_with_weights.csv",
               header = TRUE)

head(df_final_merged)

In [ ]:
df_final_merged$teeth_category = factor(df_final_merged$teeth_category,
                              levels = c("Nearly Complete (20-32)", "Moderate Loss (10-19)",
                                         "Severe Loss (1-9)", "Edentulous"),
                              labels = c("20 teeth or more", "10-19 teeth",
                                          "1-9 teeth", "Edentulous"))

table(df_final_merged$teeth_category)

In [ ]:
df_final_merged$gnri_binary <- factor(df_final_merged$gnri_binary, 
                        levels = c("High-GNRI", "Low-GNRI"))
table(df_final_merged$gnri_binary)
levels(df_final_merged$gnri_binary)

In [ ]:
# Recoding all features with consistent approach

df <- df_final_merged %>%
  mutate(
      
    # Demographic variables
    RIAGENDR = factor(RIAGENDR, levels = c(1, 2),
                    labels = c("Male", "Female")),
    
    DMDEDUC2 = factor(DMDEDUC2, levels = 1:5,
                     labels = c("Less than 9th grade", "9-11th grade",
                                "High school graduate",
                                "Some college/AA degree",
                                "College graduate or above"),
                     ordered = TRUE),
    
    RIDRETH1 = factor(RIDRETH1, levels = 1:5,
                      labels = c("Mexican American", "Other Hispanic",
                                "Non-Hispanic White", "Non-Hispanic Black",
                                "Other Race")),
    
    # Lifestyle variables - Set "No" as reference
    SMQ020 = factor(SMQ020, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Smoking status
    
    ALQ101 = factor(ALQ101, levels = c(2, 1),
                   labels = c("Under 12 drinks/1 yr", "Over 12 drinks/1 yr")),  # Alcohol
    
    # Medical conditions - all with "No" as reference level
    MCQ160B = factor(MCQ160B, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Heart failure
    
    MCQ160C = factor(MCQ160C, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Coronary heart disease
    
    MCQ160D = factor(MCQ160D, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Angina
    
    MCQ160E = factor(MCQ160E, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Heart attack
    
    MCQ160F = factor(MCQ160F, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Stroke
    
    MCQ220 = factor(MCQ220, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Cancer
    
    MCQ160L = factor(MCQ160L, levels = c(2, 1),
                    labels = c("No", "Yes")),  # Liver condition
    
    BPQ020 = factor(BPQ020, levels = c(2, 1),
                   labels = c("No", "Yes")),  # Hypertension
    
    # Multi-categorical - set "No" as reference
    DIQ010 = factor(DIQ010, levels = c(2, 1, 3),
                   labels = c("No", "Yes", "Borderline"))  # Diabetes
  )

# Verify transformations and check for any issues
summary(df[, c("RIAGENDR", "DMDEDUC2", "RIDRETH1", "SMQ020", "ALQ101", 
               "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220", "MCQ160L",
               "BPQ020", "DIQ010")])

### Crude & Adjusted WHEIGHTED Logistic Regression Analysis

In [ ]:
# function to performe weighted logistic regression for each outcome

options(survey.lonely.psu = "adjust")
options(survey.adjust.domain.lonely = TRUE)

run_weighted_logistic_analysis <- function(df, outcome) {
  results <- list()
  
  df$teeth_category <- factor(df$teeth_category, 
                              levels = c("20 teeth or more", "10-19 teeth", "1-9 teeth", "Edentulous"))
  
  for (teeth_type in c("10-19 teeth", "1-9 teeth", "Edentulous")) {
    subset_df <- df[df$teeth_category %in% c("20 teeth or more", teeth_type), ]
    
    subset_df$teeth_binary <- ifelse(subset_df$teeth_category == teeth_type, 1, 0)
    subset_df$teeth_binary <- factor(subset_df$teeth_binary, levels = c(0, 1))
    
    subset_design <- survey::svydesign(
      id = ~SDMVPSU,
      strata = ~SDMVSTRA,
      weights = ~wt,
      nest = TRUE,
      data = subset_df
    )
    
    # Crude Models
    model_crude <- survey::svyglm(
      as.formula(paste(outcome, "~ teeth_binary")),
      family = quasibinomial(),
      design = subset_design
    )
    
    # Adjusted blocks
    # Block 1: Demographics
    demo_vars <- "RIDAGEYR + RIAGENDR"
    demo_formula <- paste(outcome, "~ teeth_binary +", demo_vars)
    
    # Block 2: + Socio-Economic
    socio_vars <- "INDFMPIR + DMDEDUC2"
    socio_formula <- paste(outcome, "~ teeth_binary +", demo_vars, "+", socio_vars)
    
    # Block 3: + Behavioural
    behav_vars <- "SMQ020 + ALQ101"
    behav_formula <- paste(outcome, "~ teeth_binary +", demo_vars, "+", socio_vars, "+", behav_vars)
    
    # Block 4: + Comorbidity
    comor_vars <- "DIQ010 + BPQ020 + MCQ160B + MCQ160C + MCQ160E + MCQ160F + MCQ160L + MCQ220"
    comor_formula <- paste(outcome, "~ teeth_binary +", demo_vars, "+", socio_vars, "+", behav_vars, "+", comor_vars)
    
    # Adjusted Models
    model_demo <- survey::svyglm(as.formula(demo_formula), family = quasibinomial(), design = subset_design)
    model_socio <- survey::svyglm(as.formula(socio_formula), family = quasibinomial(), design = subset_design)
    model_behav <- survey::svyglm(as.formula(behav_formula), family = quasibinomial(), design = subset_design)
    model_comor <- survey::svyglm(as.formula(comor_formula), family = quasibinomial(), design = subset_design)
    
    extract_results <- function(model) {
      summ <- summary(model)
      coefs <- coef(summ)
      idx <- which(rownames(coefs) == "teeth_binary1")
      ci <- exp(confint(model)[idx,])
      
      if (length(idx) > 0) {
        result <- list(
          OR = exp(coefs[idx, "Estimate"]),
          # Using Wald's test
          #CI_Lower = exp(coefs[idx, "Estimate"] - 1.96 * coefs[idx, "Std. Error"]),
          #CI_Upper = exp(coefs[idx, "Estimate"] + 1.96 * coefs[idx, "Std. Error"]),

          # Using Profile likelihood CI
          CI_Lower = ci[1],
          CI_Upper = ci[2],
          P_Value = coefs[idx, "Pr(>|t|)"]
        )
      } else {
        result <- list(
          OR = NA,
          CI_Lower = NA,
          CI_Upper = NA,
          P_Value = NA
        )
      }
      
      return(result)
    }
    
    results[[teeth_type]] <- list(
      crude = list(
        overall = extract_results(model_crude)
      ),
      adjusted = list(
        demo = list(
          overall = extract_results(model_demo)
        ),
        socio = list(
          overall = extract_results(model_socio)
        ),
        behav = list(
          overall = extract_results(model_behav)
        ),
        comor = list(
          overall = extract_results(model_comor)
        )
      )
    )
  }
  
  return(results)
}

In [ ]:
# Perform Crude & Adjusted Weighted Logistic Regression

outcomes <- c("gnri_binary")

results_list <- setNames(
  lapply(outcomes, function(out) run_weighted_logistic_analysis(df, out)),
  outcomes
)

In [ ]:
# function to create a table using regression results

create_summary_table <- function(results_list, outcome_labels) {
  library(gtsummary)
  library(gt)
  library(dplyr)
  library(tidyr)
  
  format_or_ci_pval <- function(or, ci_low, ci_high, p_val) {
    stars <- ""
    if (p_val < 0.001) stars <- "***"
    else if (p_val < 0.01) stars <- "**"
    else if (p_val < 0.05) stars <- "*"
    
    formatted <- sprintf("%.2f [%.2f-%.2f]%s", or, ci_low, ci_high, stars)
    return(formatted)
  }
  
  result_df <- data.frame(
    outcome = character(),
    model = character(),
    teeth_10_19 = character(),
    teeth_1_9 = character(),
    teeth_edentulous = character(),
    stringsAsFactors = FALSE
  )
  
  for (i in seq_along(results_list)) {
    outcome_name <- names(results_list)[i]
    outcome_label <- outcome_labels[i]
    result <- results_list[[outcome_name]]
    
    # Model Crude
    row_crude <- data.frame(
      outcome = outcome_label,
      model = "Crude model",
      teeth_10_19 = format_or_ci_pval(
        result[["10-19 teeth"]]$crude$overall$OR,
        result[["10-19 teeth"]]$crude$overall$CI_Lower,
        result[["10-19 teeth"]]$crude$overall$CI_Upper,
        result[["10-19 teeth"]]$crude$overall$P_Value
      ),
      teeth_1_9 = format_or_ci_pval(
        result[["1-9 teeth"]]$crude$overall$OR,
        result[["1-9 teeth"]]$crude$overall$CI_Lower,
        result[["1-9 teeth"]]$crude$overall$CI_Upper,
        result[["1-9 teeth"]]$crude$overall$P_Value
      ),
      teeth_edentulous = format_or_ci_pval(
        result[["Edentulous"]]$crude$overall$OR,
        result[["Edentulous"]]$crude$overall$CI_Lower,
        result[["Edentulous"]]$crude$overall$CI_Upper,
        result[["Edentulous"]]$crude$overall$P_Value
      ),
      stringsAsFactors = FALSE
    )
    
    # Model 1: Demographics
    row_demo <- data.frame(
      outcome = "",
      model = "Model 1: Demographics",
      teeth_10_19 = format_or_ci_pval(
        result[["10-19 teeth"]]$adjusted$demo$overall$OR,
        result[["10-19 teeth"]]$adjusted$demo$overall$CI_Lower,
        result[["10-19 teeth"]]$adjusted$demo$overall$CI_Upper,
        result[["10-19 teeth"]]$adjusted$demo$overall$P_Value
      ),
      teeth_1_9 = format_or_ci_pval(
        result[["1-9 teeth"]]$adjusted$demo$overall$OR,
        result[["1-9 teeth"]]$adjusted$demo$overall$CI_Lower,
        result[["1-9 teeth"]]$adjusted$demo$overall$CI_Upper,
        result[["1-9 teeth"]]$adjusted$demo$overall$P_Value
      ),
      teeth_edentulous = format_or_ci_pval(
        result[["Edentulous"]]$adjusted$demo$overall$OR,
        result[["Edentulous"]]$adjusted$demo$overall$CI_Lower,
        result[["Edentulous"]]$adjusted$demo$overall$CI_Upper,
        result[["Edentulous"]]$adjusted$demo$overall$P_Value
      ),
      stringsAsFactors = FALSE
    )
    
    # Model 2: Socio-economic
    row_socio <- data.frame(
      outcome = "",
      model = "Model 2: + Socioeconomic",
      teeth_10_19 = format_or_ci_pval(
        result[["10-19 teeth"]]$adjusted$socio$overall$OR,
        result[["10-19 teeth"]]$adjusted$socio$overall$CI_Lower,
        result[["10-19 teeth"]]$adjusted$socio$overall$CI_Upper,
        result[["10-19 teeth"]]$adjusted$socio$overall$P_Value
      ),
      teeth_1_9 = format_or_ci_pval(
        result[["1-9 teeth"]]$adjusted$socio$overall$OR,
        result[["1-9 teeth"]]$adjusted$socio$overall$CI_Lower,
        result[["1-9 teeth"]]$adjusted$socio$overall$CI_Upper,
        result[["1-9 teeth"]]$adjusted$socio$overall$P_Value
      ),
      teeth_edentulous = format_or_ci_pval(
        result[["Edentulous"]]$adjusted$socio$overall$OR,
        result[["Edentulous"]]$adjusted$socio$overall$CI_Lower,
        result[["Edentulous"]]$adjusted$socio$overall$CI_Upper,
        result[["Edentulous"]]$adjusted$socio$overall$P_Value
      ),
      stringsAsFactors = FALSE
    )
    
    # Model 3: Behavioral
    row_behav <- data.frame(
      outcome = "",
      model = "Model 3: + Behavioral",
      teeth_10_19 = format_or_ci_pval(
        result[["10-19 teeth"]]$adjusted$behav$overall$OR,
        result[["10-19 teeth"]]$adjusted$behav$overall$CI_Lower,
        result[["10-19 teeth"]]$adjusted$behav$overall$CI_Upper,
        result[["10-19 teeth"]]$adjusted$behav$overall$P_Value
      ),
      teeth_1_9 = format_or_ci_pval(
        result[["1-9 teeth"]]$adjusted$behav$overall$OR,
        result[["1-9 teeth"]]$adjusted$behav$overall$CI_Lower,
        result[["1-9 teeth"]]$adjusted$behav$overall$CI_Upper,
        result[["1-9 teeth"]]$adjusted$behav$overall$P_Value
      ),
      teeth_edentulous = format_or_ci_pval(
        result[["Edentulous"]]$adjusted$behav$overall$OR,
        result[["Edentulous"]]$adjusted$behav$overall$CI_Lower,
        result[["Edentulous"]]$adjusted$behav$overall$CI_Upper,
        result[["Edentulous"]]$adjusted$behav$overall$P_Value
      ),
      stringsAsFactors = FALSE
    )
    
    # Model 4: Comorbidities
    row_comor <- data.frame(
      outcome = "",
      model = "Model 4: + Comorbidities",
      teeth_10_19 = format_or_ci_pval(
        result[["10-19 teeth"]]$adjusted$comor$overall$OR,
        result[["10-19 teeth"]]$adjusted$comor$overall$CI_Lower,
        result[["10-19 teeth"]]$adjusted$comor$overall$CI_Upper,
        result[["10-19 teeth"]]$adjusted$comor$overall$P_Value
      ),
      teeth_1_9 = format_or_ci_pval(
        result[["1-9 teeth"]]$adjusted$comor$overall$OR,
        result[["1-9 teeth"]]$adjusted$comor$overall$CI_Lower,
        result[["1-9 teeth"]]$adjusted$comor$overall$CI_Upper,
        result[["1-9 teeth"]]$adjusted$comor$overall$P_Value
      ),
      teeth_edentulous = format_or_ci_pval(
        result[["Edentulous"]]$adjusted$comor$overall$OR,
        result[["Edentulous"]]$adjusted$comor$overall$CI_Lower,
        result[["Edentulous"]]$adjusted$comor$overall$CI_Upper,
        result[["Edentulous"]]$adjusted$comor$overall$P_Value
      ),
      stringsAsFactors = FALSE
    )
    
    # Empty line after each outcome
    if (i < length(results_list)) {
      row_empty <- data.frame(
        outcome = "",
        model = "",
        teeth_10_19 = "",
        teeth_1_9 = "",
        teeth_edentulous = "",
        stringsAsFactors = FALSE
      )
      
      result_df <- rbind(result_df, row_crude, row_demo, row_socio, row_behav, row_comor, row_empty)
    } else {
      result_df <- rbind(result_df, row_crude, row_demo, row_socio, row_behav, row_comor)
    }
  }
  
  gt_table <- gt(result_df) %>%
    tab_header(
      title = "Association Between Dental Status and Geriatric Nutritional Risk Index (GNRI)",
      subtitle = "Odds Ratios with 95% Confidence Intervals from Weighted Logistic Regression Models"
    ) %>%
    cols_label(
      outcome = "GNRI",
      model = "Model Adjustment",
      teeth_10_19 = "10-19 teeth",
      teeth_1_9 = "1-9 teeth", 
      teeth_edentulous = "Edentulous"
    ) %>%
    tab_spanner(
      label = "Dental Status (reference: 20 or more teeth)",
      columns = c(teeth_10_19, teeth_1_9, teeth_edentulous)
    ) %>%
    tab_style(
      style = cell_text(weight = "bold"),
      locations = cells_column_labels()
    ) %>%
    tab_style(
      style = cell_text(weight = "bold"),
      locations = cells_body(
        columns = outcome,
        rows = outcome != ""
      )
    ) %>%
    tab_footnote(
      footnote = "* p < 0.05, ** p < 0.01, *** p < 0.001",
      locations = cells_column_labels(columns = teeth_10_19)
    ) %>%
    tab_footnote(
      footnote = "OR [95% CI] presented",
      locations = cells_column_labels(columns = teeth_1_9)
    ) %>%
    tab_footnote(
      footnote = "Models: Crude = unadjusted; Model 1 = age, gender; Model 2 = Model 1 + PIR, education; Model 3 = Model 2 + smoking, alcohol; Model 4 = Model 3 + comorbidities",
      locations = cells_column_labels(columns = model)
    ) %>%
    opt_row_striping() %>%
    tab_options(
      row.striping.include_table_body = TRUE,
      column_labels.background.color = "lightgray",
      heading.title.font.size = "large",
      heading.subtitle.font.size = "medium",
      table.width = pct(100),
      table.border.top.style = "solid",
      table.border.top.width = px(2),
      table.border.bottom.style = "solid",
      table.border.bottom.width = px(2)
    )
  
  return(gt_table)
}

In [ ]:
# Create and visualize table
# Remember to check how you insert outcomes in the outcomes vector before run regression analysis few cells above

outcome_labels <- c(
    "gnri_binary"
)

summary_table <- create_summary_table(results_list, outcome_labels)
summary_table$`_data`

In [ ]:
# Save results in a docx table

library(officer)
library(flextable)
library(gt)

save_gt_as_docx <- function(gt_table, filename) {
  library(officer)
  library(flextable)
  
  df <- gt_table$`_data`
  
  ft <- flextable::flextable(df)
  
  # Format
  ft <- ft %>%
    flextable::autofit() %>%
    flextable::theme_vanilla() %>%
    flextable::fontsize(size = 10) %>%
    flextable::padding(padding = 3)
  
  doc <- officer::read_docx()
  
  # Title
  doc <- doc %>% 
    officer::body_add_par("Association Between Dental Status and Geriatric Nutritional Risk Index (GNRI)", style = "heading 1") %>%
    officer::body_add_par("Odds Ratios with 95% Confidence Intervals from Weighted Logistic Regression Models", style = "heading 2") %>%
    officer::body_add_par("", style = "Normal")
  
  # Table
  doc <- doc %>% 
    flextable::body_add_flextable(ft)
  
  # Notes
  doc <- doc %>%
    officer::body_add_par("", style = "Normal") %>%
    officer::body_add_par("* p < 0.05, ** p < 0.01, *** p < 0.001", style = "Normal") %>%
    officer::body_add_par("OR [95% CI] presented", style = "Normal") %>%
    officer::body_add_par("Models: Crude = unadjusted; Model 1 = age, gender; Model 2 = Model 1 + PIR, education, ethnicity; Model 3 = Model 2 + smoking, alcohol; Model 4 = Model 3 + comorbidities", style = "Normal")
  
  print(doc, target = filename)
  
  message(paste("Table saved to", filename))
}

save_gt_as_docx(summary_table, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/results/NHANES_09_14_GNRI_teeth/nhanes_5_regression_teeth_gnri.docx")